# Mini Project 9 — Retail Orders: Bronze to Silver (Practice / Review)

**Role:** Data Engineer at a small online retail company.

**Scenario:** The sales team exported one month of orders from an old system into a single CSV file (`retail_orders_dirty.csv`). The file passed through several people, so the column names are inconsistent and every value came out as raw text. Before anyone can trust these numbers, they need a clean, typed table they can query.

Your job: turn this raw file into a reliable **silver** table. This project is a **focused review** — the main muscles are **type conversion** and **column renaming**, plus the cleaning steps around them.

---

### Data dictionary (business meaning — the raw headers are messy on purpose)

| Business field | What it means |
|---|---|
| order id | Unique id of the order line |
| customer name | Name of the customer |
| product | Product name |
| category | Product category |
| quantity | How many units were ordered |
| unit price | Price of one unit (in the local currency) |
| order date | The day the order was placed |
| status | Order status (completed / shipped / pending / cancelled) |
| discount | Discount applied to the order |

---

### Rules of this exercise
1. **Profile before you change anything.** Look at the data first, decide, then act.
2. Work in layers: **bronze** (raw copy) → **silver** (clean + typed).
3. Every decision is a business decision — if you drop or change something, be ready to say **why**.
4. Answer the interview questions in your own words (English). Keep them short.

> No step-by-step recipe and no function names are given on purpose. You decide **what** each requirement needs and **how** to do it. If you get stuck, ask for one hint — not the answer.

---

**Setup:** Upload `retail_orders_dirty.csv` to a Volume, then set the path below.

In [ ]:
# Set your file path (change to where you uploaded the CSV)
raw_path = "/Volumes/dev/spark_db/datasets/mini-projects/raw_data/retail_orders_dirty.csv"


## R1 — Bronze: load the raw file exactly as it is

Read the CSV into a DataFrame **without letting Spark guess any types** — every column should come in as text. This is your untouched bronze copy: no cleaning yet.

Then take a first look at the data and the schema so you understand what you are dealing with.

**Deliverable:** a bronze DataFrame with the header row respected, all columns as text, and you have looked at the rows + schema.

In [ ]:
orders_retail_raw_df = (
    spark.read
    .option("header", True)
    .format("csv")
    .load(raw_path)
)
orders_retail_raw_df.display()


### R1 — Interview questions
1. Why do we read every column as text in the bronze layer instead of letting Spark infer the types?
2. What is the risk of letting Spark automatically infer types on a dirty file like this?
3. What is the difference between the bronze layer and the silver layer?

1. If we use inferSchema and if any column has even one broken value, Spark can guess that column wrong. For example if we have date column and we use inferschema, if we have even one integer or one string spark makes this column string and preferably we transform every columns an string first and we clean columns then we transform column into their real types.
2. I already answered in answer one.
3. Bronze layer means raw data but silver layer means cleaned - transformed data. We clean bronze data to make it silver layer.


## R2 — Fix the column names

The raw headers are a mess: mixed case, extra spaces, and inconsistent styles (look closely at each header). Rename every column to a clean, consistent, code-friendly name (lower case, English, no spaces).

**Deliverable:** a DataFrame where all nine columns have clean, consistent names.

> Trap to watch for: some header names have leading/trailing spaces that you cannot see at a glance. A rename that looks correct can silently do nothing if the source name does not match exactly.

In [ ]:
orders_retail_raw_df.columns

In [ ]:
orders_retail_raw_df = orders_retail_raw_df.withColumnsRenamed({
    "Order ID" : "order_id",
    "cust_NAME" : "customer_name",
    " Product "   : "product",
    "Category"  : "category",
    "Qty"       : "quantity",
    "STATUS"    : "status",
    "Unit Price": "unit_price",
    "order date": "order_date",
    
})

orders_retail_raw_df.display()


### R2 — Interview questions
1. If your rename runs without error but the column name does not change, what is the most likely cause?
2. When you need to rename many columns at once, what is a clean way to do it (describe the idea, not the exact code)?
3. Why do good column names matter for the people who will query the silver table later?

1. We didn't assign the result to a new dataframe (with the same or a different name). 
2. we generally use withColumnsRenamed method ( which I used for our dataset.)
3. Because what those columns represent should be understandable easily to do correct analyze and calculations.


## R3 — Turn junk placeholders into real nulls

Some columns contain placeholder text that is not a real value — for example things like `ERROR`, `UNKNOWN`, and `N/A`. These are not missing-in-a-clean-way; they are junk strings pretending to be data.

Find where these appear and replace them with a true missing value, so later steps and counts behave correctly.

**Deliverable:** a DataFrame where placeholder junk is now a real missing value, not text.

> Think first: a filter that checks for "missing" will **not** catch a cell that literally contains the text `ERROR` — because that cell is full, not empty.

In [ ]:
orders_retail_raw_df.display()


In [ ]:
from pyspark.sql.functions import col 

print(orders_retail_raw_df.filter(col("order_id").isin("ERROR","UNKNOWN","N/A")).count())
print(orders_retail_raw_df.filter(col("customer_name").isin("ERROR","UNKNOWN","N/A")).count())
print(orders_retail_raw_df.filter(col("product").isin("ERROR","UNKNOWN","N/A")).count())
print(orders_retail_raw_df.filter(col("category").isin("ERROR","UNKNOWN","N/A")).count())
print(orders_retail_raw_df.filter(col("quantity").isin("ERROR","UNKNOWN","N/A")).count())
print(orders_retail_raw_df.filter(col("unit_price").isin("ERROR","UNKNOWN","N/A")).count())
print(orders_retail_raw_df.filter(col("order_date").isin("ERROR","UNKNOWN","N/A")).count())
print(orders_retail_raw_df.filter(col("status").isin("ERROR","UNKNOWN","N/A")).count())
print(orders_retail_raw_df.filter(col("discount").isin("ERROR","UNKNOWN","N/A")).count())







In [ ]:
from pyspark.sql.functions import expr
order_retail_raw_nulled_df = orders_retail_raw_df.withColumns({
    "quantity" : expr("CASE WHEN quantity IN ('ERROR','UNKNOWN','N/A') THEN NULL ELSE quantity END"),
    "unit_price": expr("case when unit_price in ('ERROR','UNKNOWN','N/A') then null else unit_price end"),
    "discount"  : expr("case when discount in ('ERROR','UNKNOWN','N/A') then null else discount end")
})

order_retail_raw_nulled_df.display()

In [ ]:
from pyspark.sql.functions import col 

print(order_retail_raw_nulled_df.filter(col("order_id").isin("ERROR","UNKNOWN","N/A")).count())
print(order_retail_raw_nulled_df.filter(col("customer_name").isin("ERROR","UNKNOWN","N/A")).count())
print(order_retail_raw_nulled_df.filter(col("product").isin("ERROR","UNKNOWN","N/A")).count())
print(order_retail_raw_nulled_df.filter(col("category").isin("ERROR","UNKNOWN","N/A")).count())
print(order_retail_raw_nulled_df.filter(col("quantity").isin("ERROR","UNKNOWN","N/A")).count())
print(order_retail_raw_nulled_df.filter(col("unit_price").isin("ERROR","UNKNOWN","N/A")).count())
print(order_retail_raw_nulled_df.filter(col("order_date").isin("ERROR","UNKNOWN","N/A")).count())
print(order_retail_raw_nulled_df.filter(col("status").isin("ERROR","UNKNOWN","N/A")).count())
print(order_retail_raw_nulled_df.filter(col("discount").isin("ERROR","UNKNOWN","N/A")).count())

### R3 — Interview questions
1. Why is a placeholder like the text `UNKNOWN` more dangerous than an actually-empty cell?
2. What is the difference between *filling* a missing value and *turning a junk value into* a missing value? (These go in opposite directions.)
3. Should placeholder cleaning happen before or after type conversion? Why?

1. Because when we do calculation with some functions like group by it could be perceived as a group by Spark, so it would be deceptive.
2. When we fill the missing value with null it doesn't affect our calculations so we still have proper numbers in our analyzes.
3. we should clean placeholder before type conversion because if we don't clean them our analyses would be deceptive because they are perceived as a separate group by PySpark.


## R4 — Convert every column to its correct type (the core of this project)

Right now everything is text. Give each column the type it should really have. Three columns need real thought:

- **unit price** — the values are text that mix a currency symbol and a currency code with the number (look at the raw values). You must get to a clean numeric type suitable for money. Decide: what do you clean first, and what numeric type is right for money vs. an approximate type?
- **order date** — the dates arrived in **several different formats** in the same column (again, look at the raw values). You need one real date type for all of them. A single fixed format will not parse them all.
- **quantity** — should be a whole number.

The remaining columns should each get a sensible type too (think about which ones are ids vs. numbers vs. text).

**Deliverable:** a fully typed DataFrame — no more "everything is a string".

> Two traps: (1) you cannot cast text like `₺199.90` straight to a number — clean the text first, then convert. (2) A strict date conversion on an unexpected format can either error out or silently produce a missing value. Decide how you want to handle the mixed formats.

In [ ]:
order_retail_raw_nulled_df.limit(3).display()

In [ ]:
order_retail_raw_nulled_df.printSchema()


In [ ]:
order_retail_raw_nulled_df.display()

In [ ]:
from pyspark.sql.functions import col, try_to_date, replace,trim, coalesce,lit

order_retail_typed_df = order_retail_raw_nulled_df.withColumns({
    "quantity" : col('quantity').cast('integer'),
    "unit_price" : trim(replace(replace(replace(col("unit_price"),lit('₺'),lit('')),lit('TL'),lit('')),lit('tl'),lit(''))).cast("decimal(10,2)"),
    "order_date" : coalesce(
        try_to_date('order_date', 'yyyy-MM-dd'),
        try_to_date('order_date', 'dd/MM/yyyy'),
        try_to_date('order_date', 'dd.MM.yyyy'),
        try_to_date('order_date', 'MM.dd.yyyy'),
        try_to_date('order_date','MMMM dd yyyy'),
        try_to_date('order_date', 'yyyy/MM/dd')
    ),
    "discount" : trim(replace(col("discount"),lit('%'),lit(''))).cast("decimal(10,2)")

})

order_retail_typed_df.display()

In [ ]:

print(order_retail_raw_nulled_df.filter(col("unit_price").isNull()).count())
print(order_retail_raw_nulled_df.filter(col("quantity").isNull()).count())
print(order_retail_raw_nulled_df.filter(col("order_date").isNull()).count())
print(order_retail_raw_nulled_df.filter(col("discount").isNull()).count())





print(order_retail_typed_df.filter(col("unit_price").isNull()).count())
print(order_retail_typed_df.filter(col("quantity").isNull()).count())
print(order_retail_typed_df.filter(col("order_date").isNull()).count())
print(order_retail_typed_df.filter(col("discount").isNull()).count())


> **Note on this check:** on the first run the `coalesce(try_to_date ...)` list was missing some of the formats, and `order_date` jumped from **0 to 25 nulls** after casting — silent data loss, caught only by this before/after comparison. The missing formats were added one by one until the after-cast null count came back to 0, which is the state shown above. This is exactly why the null check runs on *both* sides of the cast.

### R4 — Interview questions
1. For money, which numeric type would you choose and why? What is the risk of the approximate floating-point type?
2. You have several date formats in one column. Describe your strategy to get them all into a single real date column.
3. What is the difference between a strict conversion that errors on bad input and a safe conversion that returns a missing value instead? When would you prefer each?
4. Why can't you cast `"149.90 TL"` directly to a number? What has to happen first?
5. Why is `order id` better kept as text than as a number, even though it looks numeric?

1. I chose DecimalType for money because if we choose double we can get floating-point rounding problems.
2. To convert all values into date I use coalesce and try_to_date. Coalesce checks all formats one by one and make them date format. 
3. A strict conversion throws an error directly and stops the job, while a safe conversion returns null instead. I use the safe one when data is big and messy. I use strict one when data quality is critical or if i trust data. In this example with order_date i used the safe one and I found 25 silent nulls by counting.
4. We cannot use it because spark throws an error because of 'TL' ( string part ). First I deleted 'TL' by using the replace function and then I converted that column to a proper format which is decimalType.
5. I keep the order id as text because it is an identifier, not a number I do math on. I never do math calculations on IDs, so it does not need a numeric type. Also, a numeric type would drop leading zeros and break the ID, and it can't hold IDs with letters like 'ORD-1001'.


## R5 — Normalize the text columns

The `status` and `category` columns are the same values written many different ways: different capitalization and hidden spaces (e.g. `completed`, `Completed`, `CANCELLED`, ` shipped `; `home & kitchen` vs `HOME & KITCHEN`). If you group by these columns as-is, the same real category will split into several fake buckets.

Make each of these columns consistent, so that one real value = one written value. Also clean up any stray spaces around the product names.

**Deliverable:** `status`, `category`, and `product` are clean and consistent.

In [ ]:
order_retail_typed_df.count()

In [ ]:
order_retail_typed_df.display()

In [ ]:
from pyspark.sql.functions import col, initcap, trim 

order_normalized_df = order_retail_typed_df.withColumns({
    "order_id"       : trim(col("order_id")),
    "customer_name"  : trim(initcap(col("customer_name"))),
    "product"        : trim(initcap(col("product"))),
    "status"         : trim(initcap(col("status"))),
    "category"       : trim(initcap(col("category")))
})

order_normalized_df.display()



### R5 — Interview questions
1. Why will grouping by an un-normalized `category` give you wrong totals?
2. Name two kinds of "invisible" text problems that break grouping, and how you would fix each.
3. Would you normalize text before or after removing duplicates? Explain your reasoning.

1. Because every different character is counted as different groups by Spark. For example if we have 'home & kitchen' and 'HOME & KITCHEN' they are counted as different groups by Spark.
2. I generally use trim and initcap or lower functions together so I can clean capital character problem and I clean spaces at the same time.
3. I remove duplicates after normalizing the text because as I mentioned at question one, even though values are the same, they can be considered different groups by spark.


## R6 — Business rules, duplicates, and write the silver table

Now make the table trustworthy and save it.

- **Duplicates:** the export contains some fully repeated rows. Decide how to handle them.
- **Business rules:** a `quantity` that is zero or negative is not a valid order line. Decide what to do with those rows and be ready to defend it (dropping a row is a business decision, not a reflex).
- **Write:** save the result as a **silver** table. Make the write **idempotent** — running the notebook twice must not create duplicates or double the data.

**Deliverable:** a clean, typed, de-duplicated silver table saved to the catalog.

> Note: a missing (null) quantity is *not* the same as an invalid (negative) quantity. Missing means "we don't know"; negative means "this is wrong". Treat them as separate decisions.

In [ ]:
order_normalized_df.display()


In [ ]:
# No completely duplicated rows.

print(order_normalized_df.count())
print(order_normalized_df.dropDuplicates().count())

In [ ]:
print(order_normalized_df.select("quantity").count())
print(order_normalized_df.filter(col("quantity")<=0).count())
print(order_normalized_df.filter(col("quantity").isNull()).count())

In [ ]:
order_clean_df = order_normalized_df.filter((col("quantity").isNull() | (col("quantity")>0)))
order_clean_df.display()

### Final checks and the idempotent silver write

Before saving: confirm the business rule holds, look at the final row count, and compare the distinct `order_id` count with the total (grain check). Then write the silver table with `overwrite`, so re-running the notebook never doubles the data.

In [ ]:
# Final quality checks before writing
print(order_clean_df.count())                               # final row count (55 - 7 invalid rows = 48)
print(order_clean_df.filter(col("quantity") <= 0).count())  # business rule holds -> expect 0
print(order_clean_df.select("order_id").distinct().count()) # compare with row count: one row = one order id

In [ ]:
# Idempotent write: overwrite replaces the table, so re-running never appends duplicates
(order_clean_df.write
    .mode("overwrite")
    .saveAsTable("retail_orders_silver"))

### R6 — Interview questions
1. What does *idempotent* mean for a write, and how do you make a table write idempotent?
2. When you remove fully-duplicated rows, can you always predict which copy stays? Does it matter here?
3. You dropped rows with quantity <= 0. How would you defend that choice to a stakeholder? Is there a case where you'd keep them instead?
4. Why treat a null quantity and a negative quantity as two different problems?

1. Idempotent means running the write many times gives the same result — no duplicated or doubled data. I make it idempotent by using mode("overwrite") instead of append, so each run replaces the table instead of adding to it
2. No, I cannot predict which copy stays — it is non-deterministic. But here it does not matter, because the rows are fully identical in every column, so whichever copy stays, the result is exactly the same.
3. A valid order line must have at least one item, so zero or negative quantity is not a real sale — likely a data error. Dropping them keeps totals and revenue correct. But I would keep them if a negative quantity means a return/refund that the business wants to track — then I'd flag them, not delete them.
4. A null quantity means 'we don't know' — the data is missing and might be fixed later. A negative quantity means 'we know the value and it is wrong'. One is missing, the other is invalid — different meaning, so different action: I keep nulls but drop negatives.

## Final defense round (like an interviewer reviewing your code)

Answer these in the cell below, in English:

1. Walk me through your pipeline in three sentences: bronze, then what, then silver.
2. Which single step in this project was the most error-prone, and why?
3. If tomorrow the same export arrives with a **new** date format you have never seen, does your pipeline break? What happens to those rows, and how would you find them?
4. How would you prove to me that your silver table is actually clean — what checks would you run?
5. Did the order of your steps matter (rename → placeholders → types → normalize → filter)? Could any two be swapped safely, and which could not?

1. Bronze is the raw data, untouched, and the cleaned and transformed data (ready to analyze) is silver layer.
2. For this project transformation data to make silver layer was the hardest part especially formatting date column. I had 5 or 6 types of date so I used coalesce and try_to_date to parse them. I got lots of error because of having different types of date formats.
3. I would check coalesce part because coalesce function makes the date null if we don't have that format inside of the function. So I would check with isnull() if I have null date or not ( if we have new format ). If we have new format, I would put it inside of coalesce by using try_to_date function.
4. I would run several checks to prove the silver table is clean:

Null check — I compare the null count of each column before and after casting, so I know my type conversion did not silently drop values. Today order_date went from 0 to 25 nulls, which showed me a missing date format.
Granularity check — I compare the distinct count of the key column with the total row count to prove one row = one order line.
Invalid values — I check that no rows break the business rules, for example quantity <= 0 should return zero after cleaning.
Consistency — I group by the text columns to make sure each real value is one bucket, not split by different spelling.
Idempotency — I run the write twice and confirm the row count stays the same, not doubled.

5. Yes, the order matters in one place. Cleaning the text and casting the types cannot be swapped: if I cast first, dirty values like '$199' or '₺199' can't become a number, so they turn into null. That means silent data loss and wrong analysis. So I must clean first, then cast.
But some steps are safe to swap — for example normalize (fixing text case/spaces) and filter (dropping quantity <= 0) are independent, so their order does not change the result.
The rule: any step that another step depends on must come first (clean before cast, normalize before grouping); independent steps can go in any order.



---

## Key Takeaways

- **Bronze reads everything as text.** One broken value is enough to make `inferSchema` guess a whole column wrong — read as string, clean, then cast on your own terms.
- **A rename can silently do nothing.** Hidden spaces in headers (`' Product '`) mean the source name must match exactly — check `df.columns` before renaming, and verify after.
- **Placeholders before types.** `ERROR` / `UNKNOWN` / `N/A` are full cells, not empty ones — a null check will not find them. Turn them into real `NULL` *before* casting.
- **Clean first, then cast.** `₺199.90` / `199.90 TL` cannot become a number directly — strip the symbols, then cast. Money gets `decimal(10,2)`, never a float.
- **Mixed date formats** → `coalesce(try_to_date(col, fmt1), try_to_date(col, fmt2), ...)`. A safe cast fails *silently*, so always compare null counts **before vs. after** casting — during this run a missing format showed up as 25 new nulls and was fixed by adding formats until the count returned to 0.
- **Normalize text before grouping or dedup.** `completed` / `Completed` / `CANCELLED` split one real value into fake buckets.
- **Missing ≠ invalid.** A null quantity ("we don't know") and a `quantity <= 0` ("we know it's wrong") are two different problems — keep the nulls, drop the invalids, and be ready to defend both.
- **Idempotent write.** `mode("overwrite")` + `saveAsTable` — running the notebook twice produces the same table, not doubled data.
